In [1]:
import torch
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder
from ollama import Client as OllamaClient
from datetime import datetime
from collections import defaultdict

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {DEVICE}")

TOP_K_RETRIEVAL = 100
TOP_K_RERANK    = 15
TOP_K_PER_CAT   = 5

# Modelo local via Ollama
# Opciones según tu RAM:
#   8GB  RAM → "deepseek-r1:7b"
#   16GB RAM → "deepseek-r1:14b"  (mejor razonamiento)
#   32GB RAM → "deepseek-r1:32b"  (muy bueno)
OLLAMA_MODEL = "deepseek-r1:14b"   # ← cambia según tu RAM

ollama_client = OllamaClient(host="http://localhost:11434")

print("Configuración cargada ✓")
print(f"Modelo LLM: {OLLAMA_MODEL}")

c:\Users\josit\CUARTO CURSO\TFG\TFG_Crypto_DEF\env_TFG_DEF\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dispositivo: cuda
Configuración cargada ✓
Modelo LLM: deepseek-r1:14b


In [2]:
# Qwen3-Embedding-8B: líder MTEB multilingüe.
# Primera vez descarga ~5GB. Si no tienes GPU usa "all-MiniLM-L6-v2" (más ligero).

print("Cargando embedding model... (primera vez ~5 min)")

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2",   # 90MB, descarga en segundos
    device=DEVICE,
)

def embed_texts(texts: list[str], role: str = "query") -> list[list[float]]:
    """
    Convierte textos en vectores.
    role="query"   → para búsquedas
    role="passage" → para indexar documentos
    """
    return embedding_model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
        prompt_name=role
    ).tolist()

test = embed_texts(["prueba"])
print(f"Embedding model cargado ✓  |  Dimensiones: {len(test[0])}")

Cargando embedding model... (primera vez ~5 min)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4666.73it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 1/1 [00:00<00:00,  1.72it/s]

Embedding model cargado ✓  |  Dimensiones: 384


In [3]:
from dotenv import load_dotenv
import os

load_dotenv()

NEWSAPI_KEY = os.getenv("NEWSAPI_KEY")

In [4]:
from newspaper import Article
import requests
import hashlib
from datetime import datetime, timedelta

def enrich_with_full_text(doc: dict, url: str) -> dict:
    try:
        art = Article(url)
        art.download()
        art.parse()
        if len(art.text) > 200:
            doc["text"] = art.text[:1000]
    except:
        pass
    return doc

def fetch_newsapi(days_back: int = 7, limit: int = 100) -> list[dict]:
    docs = []
    fecha_desde = (datetime.today() - timedelta(days=days_back)).strftime("%Y-%m-%d")
    
    queries = {
        "macro"    : "crypto regulation OR bitcoin ban OR SEC crypto OR federal reserve crypto",
        "eth"      : "ethereum OR ETH blockchain OR bitcoin halving OR crypto hack",
        "sentiment": "bitcoin price OR ethereum price OR crypto market OR fear greed crypto",
    }
    
    for categoria, query in queries.items():
        url = "https://newsapi.org/v2/everything"
        params = {
            "q"        : query,
            "from"     : fecha_desde,
            "sortBy"   : "relevancy",
            "language" : "en",
            "pageSize" : min(limit // 3, 100),
            "apiKey"   : NEWSAPI_KEY,
        }
        
        try:
            r = requests.get(url, params=params, timeout=10)
            r.raise_for_status()
            articulos = r.json().get("articles", [])
        except Exception as e:
            print(f"Error en categoría {categoria}: {e}")
            continue
        
        for art in articulos:
            titulo  = art.get("title")  or ""
            resumen = art.get("description") or ""
            texto   = f"{titulo}. {resumen}"[:500]
            url_art = art.get("url", "")
            
            if not texto.strip() or titulo == "[Removed]":
                continue
            
            try:
                fecha = art["publishedAt"][:10]
            except:
                fecha = datetime.today().strftime("%Y-%m-%d")
            
            doc_id = hashlib.md5(texto[:100].encode()).hexdigest()
            
            docs.append({
                "id"      : doc_id,
                "text"    : texto,
                "date"    : fecha,
                "source"  : art.get("source", {}).get("name", "unknown"),
                "category": categoria,
                "url"     : url_art,   # ← guardamos la URL para enriquecer
            })
        
        print(f"  [{categoria}] {len(articulos)} artículos descargados")
    
    docs = list({d["id"]: d for d in docs}.values())
    
    keywords_cripto = ["bitcoin", "ethereum", "crypto", "blockchain", "btc", "eth",
                       "defi", "stablecoin", "coinbase", "binance", "sec crypto"]
    docs = [d for d in docs if any(k in d["text"].lower() for k in keywords_cripto)]
    
    # Enriquecer con texto completo
    print(f"\nEnriqueciendo {len(docs)} artículos con texto completo...")
    for i, doc in enumerate(docs):
        if doc.get("url"):
            docs[i] = enrich_with_full_text(doc, doc["url"])
        if i % 10 == 0:
            print(f"  {i}/{len(docs)}...")
    
    return docs


docs = fetch_newsapi(days_back=7, limit=100)

print(f"\nTotal noticias: {len(docs)}")
print(f"\nEjemplo con texto completo:")
print(f"  Fuente   : {docs[0]['source']}")
print(f"  Categoría: {docs[0]['category']}")
print(f"  Texto    : {docs[0]['text'][:200]}...")

  [macro] 32 artículos descargados
  [eth] 33 artículos descargados
  [sentiment] 26 artículos descargados

Enriqueciendo 70 artículos con texto completo...
  0/70...
  10/70...
  20/70...
  30/70...
  40/70...
  50/70...
  60/70...

Total noticias: 70

Ejemplo con texto completo:
  Fuente   : CoinDesk
  Categoría: macro
  Texto    : Global liquidity is set to deteriorate sharply, according to Russell Thompson, chief investment officer at crypto asset manager Hilbert Group (HILB), who said even a quick geopolitical resolution in I...


In [5]:
import pandas as pd

df_docs = pd.DataFrame(docs)
df_docs.to_csv("noticias.csv", index=False)
df_docs.head()

,id,text,date,source,category,url
0,7d27f03df087f45ca2a06b49b6361b56,Global liquidity is set to deteriorate sharply...,2026-04-20,CoinDesk,macro,https://www.coindesk.com/markets/2026/04/20/bi...
1,426341be964f7513d28548d7d04380d5,"As readers of this newsletter may be aware, Co...",2026-04-19,CoinDesk,macro,https://www.coindesk.com/policy/2026/04/19/pre...
2,c379f97789e452c31336136cc7a911fb,"This is an edition of The Atlantic Daily, a ne...",2026-04-24,The Atlantic,macro,https://www.theatlantic.com/newsletters/2026/0...
3,e3994981e7083f5fb3200be0c5ea3e38,Crypto and crypto markets pulled back Tuesday ...,2026-04-21,CoinDesk,macro,https://www.coindesk.com/markets/2026/04/21/bi...
4,7775c39b06f14c25be8232de02f144d5,"Michael Saylor, executive chairman of Strategy...",2026-04-24,CoinDesk,macro,https://www.coindesk.com/markets/2026/04/23/mi...


In [6]:
import kagglehub
import shutil
import os

# Ruta destino
dest = r"C:\Users\josit\CUARTO CURSO\TFG\TFG_Crypto_DEF\data\news"
os.makedirs(dest, exist_ok=True)

# Descargar
path = kagglehub.dataset_download("oliviervha/crypto-news")
print("Descargado en caché:", path)

# Mover a tu ruta
for f in os.listdir(path):
    shutil.copy(os.path.join(path, f), os.path.join(dest, f))
    print(f"Copiado: {f}")

print("Listo en:", dest)

Descargado en caché: C:\Users\josit\.cache\kagglehub\datasets\oliviervha\crypto-news\versions\10
Copiado: cryptonews.csv
Listo en: C:\Users\josit\CUARTO CURSO\TFG\TFG_Crypto_DEF\data\news


In [7]:
import pandas as pd

df_news = pd.read_csv(
    r"C:\Users\josit\CUARTO CURSO\TFG\TFG_Crypto_DEF\data\news\cryptonews.csv",
    header=None   # parece que no tiene cabecera
)

print(df_news.shape)
print(df_news.head(3))
print(df_news.dtypes)

(31038, 7)
                     0                                                  1  \
0                 date                                          sentiment   
1  2023-12-19 06:40:41  {'class': 'negative', 'polarity': -0.1, 'subje...   
2  2023-12-19 06:03:24  {'class': 'neutral', 'polarity': 0.0, 'subject...   

            2           3                                                  4  \
0      source     subject                                               text   
1  CryptoNews     altcoin  Grayscale CEO Michael Sonnenshein believes the...   
2  CryptoNews  blockchain  In an exclusive interview with CryptoNews, Man...   

                                                   5  \
0                                              title   
1  Grayscale CEO Calls for Simultaneous Approval ...   
2  Indian Government is Actively Collaborating Wi...   

                                                   6  
0                                                url  
1  https://cryptonews.co

In [8]:
import pandas as pd
import ast
import hashlib

# Cargar saltando la primera fila (es la cabecera)
df_news = pd.read_csv(
    r"C:\Users\josit\CUARTO CURSO\TFG\TFG_Crypto_DEF\data\news\cryptonews.csv",
    header=0,
    names=["date", "sentiment", "source", "subject", "text", "title", "url"]
)

# Quitar la fila 0 que era la cabecera repetida
df_news = df_news[df_news["date"] != "date"].reset_index(drop=True)

# Parsear el sentimiento (viene como string de dict)
def parse_sentiment(s):
    try:
        d = ast.literal_eval(s)
        return d.get("class", "neutral")
    except:
        return "neutral"

df_news["sentiment_class"] = df_news["sentiment"].apply(parse_sentiment)

# Fecha limpia
df_news["date_clean"] = pd.to_datetime(df_news["date"], errors="coerce").dt.strftime("%Y-%m-%d")
df_news = df_news.dropna(subset=["date_clean"])

def asignar_categoria(row):
    texto_completo = (str(row["subject"]) + " " + 
                      str(row["title"]) + " " + 
                      str(row["text"])).lower()
    
    if any(w in texto_completo for w in ["sec ", "regulation", "ban ", "law ",
                                          "government", "federal reserve", "fed ",
                                          "sanction", "inflation", "tax ", "policy",
                                          "court", "legal", "congress", "senate"]):
        return "macro"
    elif any(w in texto_completo for w in ["ethereum", " eth ", "bitcoin", " btc ",
                                            "defi", "hack", "upgrade", "fork",
                                            "mining", "blockchain", "nft", "wallet"]):
        return "eth"
    else:
        return "sentiment"

# Cambiar también la llamada:
df_news["category"] = df_news.apply(asignar_categoria, axis=1)

# Convertir a formato add_documents()
def to_docs(df):
    docs = []
    for _, row in df.iterrows():
        titulo = str(row["title"]) if pd.notna(row["title"]) else ""
        texto  = str(row["text"])  if pd.notna(row["text"])  else ""
        combined = f"[{row['sentiment_class'].upper()}] {titulo}. {texto}"[:500]
        
        if not combined.strip():
            continue
        
        doc_id = hashlib.md5(combined[:100].encode()).hexdigest()
        
        docs.append({
            "id"      : doc_id,
            "text"    : combined,
            "date"    : row["date_clean"],
            "source"  : str(row["source"]),
            "category": row["category"],
        })
    
    # Eliminar duplicados
    docs = list({d["id"]: d for d in docs}.values())
    return docs

docs_historicos = to_docs(df_news)

print(f"Total documentos históricos: {len(docs_historicos)}")
print(f"\nDistribución por categoría:")
cats = {}
for d in docs_historicos:
    cats[d["category"]] = cats.get(d["category"], 0) + 1
for k, v in cats.items():
    print(f"  {k}: {v}")
print(f"\nEjemplo:")
print(f"  {docs_historicos[0]['text'][:200]}")

Total documentos históricos: 31007

Distribución por categoría:
  macro: 4713
  eth: 20221
  sentiment: 6073

Ejemplo:
  [NEGATIVE] Grayscale CEO Calls for Simultaneous Approval of Spot Products to Level the Field. Grayscale CEO Michael Sonnenshein believes the SEC needs to approve spot Bitcoin exchange-traded funds (ET


In [9]:
# Base de datos vectorial local. Guarda todo en ./chroma_db en disco.

chroma_client = chromadb.PersistentClient(path="./chroma_db")

collection = chroma_client.get_or_create_collection(
    name="crypto_news",
    metadata={"hnsw:space": "cosine"}
)

def add_documents(docs: list[dict]):
    """
    Añade documentos a ChromaDB.
    Cada doc debe tener: id, text, date (YYYY-MM-DD), source, category.
    category debe ser: "macro" | "eth" | "sentiment"
    """
    if not docs:
        return
    texts     = [d["text"] for d in docs]
    ids       = [d["id"]   for d in docs]
    metadatas = [{"date": d["date"], "source": d["source"], "category": d["category"]}
                 for d in docs]
    embeddings = embedding_model.encode(
        texts, batch_size=32, normalize_embeddings=True, prompt_name="passage"
    ).tolist()
    collection.add(ids=ids, embeddings=embeddings, documents=texts, metadatas=metadatas)
    print(f"Añadidos {len(docs)} docs. Total en BD: {collection.count()}")


def search(query: str, n_results: int = TOP_K_RETRIEVAL, date_from: str = None) -> list[dict]:
    """
    Busca los documentos más similares a la query.
    date_from: filtro opcional de fecha mínima "YYYY-MM-DD"
    """
    query_vec = embedding_model.encode(
        [query], normalize_embeddings=True, prompt_name="query"
    ).tolist()
    where   = {"date": {"$gte": date_from}} if date_from else None
    results = collection.query(
        query_embeddings=query_vec,
        n_results=min(n_results, collection.count()),
        where=where,
        include=["documents", "metadatas", "distances"]
    )
    return [
        {"text": text, "date": meta["date"], "source": meta["source"],
         "category": meta["category"], "similarity": round(1 - dist, 4)}
        for text, meta, dist in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0]
        )
    ]

print(f"ChromaDB listo ✓  |  Docs en colección: {collection.count()}")

ChromaDB listo ✓  |  Docs en colección: 0


In [10]:
RUTA_CSV = "../data/csv/df_merged.csv"

df = pd.read_csv(RUTA_CSV, parse_dates=["date"], index_col="date")
df.sort_index(inplace=True)



In [11]:
print(df.columns.tolist())
print(df.head(2))

['btc_open', 'btc_high', 'btc_low', 'btc_close', 'btc_volume', 'eth_open', 'eth_high', 'eth_low', 'eth_close', 'eth_volume', 'btc_dominance', 'eth_dominance', 'alt_dominance', 'fear_greed', 'FearGreed_Label', 'inflation', 'btc_mcap', 'eth_mcap', 'fed_rate']
                btc_open      btc_high      btc_low    btc_close   btc_volume  \
date                                                                            
2018-02-01  10237.299805  10288.799805  8812.280273  9170.540039   9959400448   
2018-02-02   9142.280273   9142.280273  7796.490234  8830.750000  12726899712   

               eth_open     eth_high    eth_low    eth_close  eth_volume  \
date                                                                       
2018-02-01  1119.369995  1161.349976  984.81897  1036.790039  5261680128   
2018-02-02  1035.770020  1035.770020  757.97998   915.784973  6713290240   

            btc_dominance  eth_dominance  alt_dominance  fear_greed  \
date                                     

In [12]:
# MVRV = Market Value / Realized Value
# Realized Value aproximado = media móvil 365d del precio de cierre
# Lógica: el precio medio de los últimos 365 días aproxima el precio
# al que el "mercado agregado" tiene su posición abierta.

df["realized_price_eth"] = df["eth_close"].rolling(730).mean()
df["mvrv_aprox"]         = df["eth_close"] / df["realized_price_eth"]

# Ver último valor
ultimo = df[["eth_close", "realized_price_eth", "mvrv_aprox"]].dropna().iloc[-1]
print(f"Precio ETH actual    : ${ultimo['eth_close']:.2f}")
print(f"Realized Price (365d): ${ultimo['realized_price_eth']:.2f}")
print(f"MVRV aproximado      : {ultimo['mvrv_aprox']:.2f}")
print()
if ultimo["mvrv_aprox"] > 3.5:
    print("Señal: TECHO — sobrecomprado históricamente")
elif ultimo["mvrv_aprox"] > 2.0:
    print("Señal: BULL MARKET — precaución creciente")
elif ultimo["mvrv_aprox"] > 1.0:
    print("Señal: NEUTRAL")
else:
    print("Señal: SUELO — infracomprado históricamente")

Precio ETH actual    : $2903.58
Realized Price (365d): $3051.65
MVRV aproximado      : 0.95

Señal: SUELO — infracomprado históricamente


In [45]:
import os
from google import genai
from dotenv import load_dotenv

load_dotenv()

# ── Leer el TXT estático ──────────────────────────────────────────────────────
with open("../data/txt/patrones1.txt", "r", encoding="utf-8") as f:
    contexto = f.read()

# ── Construir el prompt ───────────────────────────────────────────────────────
pregunta = "Explícame el patrón post-halving y en qué fase del ciclo estamos aproximadamente."

prompt = f"""
{contexto}

<DINAMICO>
Sin datos dinámicos por ahora. Razona solo con los patrones estáticos.
</DINAMICO>

<NOTICIAS>
Sin noticias por ahora.
</NOTICIAS>

<PREGUNTA>
{pregunta}
</PREGUNTA>
"""

# ── Llamar al modelo ──────────────────────────────────────────────────────────
client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

print(response.text)

Como analista cuantitativo de mercado cripto especializado en Ethereum, procederé a explicar el patrón post-halving de Bitcoin y a discutir, dentro de las limitaciones de la información disponible, en qué fase del ciclo podríamos encontrarnos.

**Razonamiento Interno:**

1.  **Identificar la pregunta:** El usuario pregunta sobre el patrón post-halving y la fase actual del ciclo.
2.  **Buscar patrón relevante:** El "BLOQUE 1. PATRON 1 - CICLOS POST-HALVING DE BITCOIN" (ID: P1) es directamente aplicable.
3.  **Explicar P1:** Detallaré la naturaleza del halving, su frecuencia, las fechas históricas, el tiempo promedio desde el halving hasta el pico del ciclo, la magnitud de los rallies y los drawdowns observados, así como las fases generales del ciclo.
4.  **Grado de evidencia:** Recordar que P1 tiene evidencia [MODERADA] y se basa en 3 ciclos observados.
5.  **Aplicar implicaciones prácticas (con cautela):** Dado que no hay datos dinámicos como `{dias_post_halving}`, no puedo aplicar los

In [ ]:
import os
import time
from google import genai
from dotenv import load_dotenv

load_dotenv()

# ── Leer el TXT estático ──────────────────────────────────────────────────────
with open("../data/txt/patrones1.txt", "r", encoding="utf-8") as f:
    contexto = f.read()

# ── Lista de preguntas ────────────────────────────────────────────────────────
preguntas = [
    # BLOQUE 1 — Hechos históricos verificables
    "¿En qué año ocurrió el halving de Bitcoin de 2020 y cuánto bajó la recompensa de bloque? ¿Qué efecto histórico tuvo en el precio en los 12 meses siguientes?",
    "¿Cuál fue el precio máximo histórico (ATH) de Ethereum en USD y en qué fecha exacta ocurrió?",
    "Durante el ciclo alcista de 2017, ¿cuál fue el rango de precio de Ethereum desde su mínimo hasta su máximo? ¿Cuánto tiempo duró aproximadamente ese movimiento?",
    "¿Cuántos halvings de Bitcoin han ocurrido hasta la fecha y cuáles son sus fechas aproximadas?",
    "¿En qué rango de precios cotizaba Bitcoin durante el 'crypto winter' de 2018-2019?",

    # BLOQUE 2 — Patrones de ciclos y correlaciones
    "Basándote únicamente en el contexto proporcionado, ¿cuál es la relación histórica entre el halving de Bitcoin y el inicio de un bull market en Ethereum? ¿Cuántos meses de desfase suele observarse?",
    "¿Existe evidencia histórica de que Ethereum supere el rendimiento de Bitcoin durante la fase tardía de un ciclo alcista? Cita ciclos concretos para justificar tu respuesta.",
    "¿Qué es el ratio ETH/BTC y qué indica históricamente cuando ese ratio sube? ¿En qué ciclos ha marcado máximos?",
    "Describe las 4 fases clásicas de un ciclo de mercado cripto (acumulación, markup, distribución, markdown) y ejemplifícalas con fechas reales de BTC o ETH.",
    "¿Por qué algunos analistas sugieren que Ethereum opera en ciclos de 8 años mientras Bitcoin opera en ciclos de 4 años? ¿Qué datos históricos apoyan o refutan esa hipótesis?",

    # BLOQUE 3 — Razonamiento con incertidumbre
    "Con la información disponible, ¿puede afirmarse que el ciclo de 4 años de Bitcoin seguirá repitiéndose en el futuro? ¿Qué factores podrían romper ese patrón?",
    "¿Es posible predecir con precisión el precio máximo de Ethereum en un ciclo dado, basándose solo en datos de ciclos anteriores? Explica las limitaciones de ese enfoque.",
    "Si el contexto proporcionado no incluye datos sobre el precio actual de Bitcoin, ¿en qué fase del ciclo podría estar el mercado según indicadores on-chain históricos? Sé explícito sobre qué información te falta para responder con certeza.",
    "¿Qué métricas on-chain históricas han sido más útiles para identificar el techo de un ciclo alcista en Bitcoin? ¿Son igualmente válidas para Ethereum?",
    "¿Cuál es la diferencia estructural entre el bull market de 2017 y el de 2021 en términos de quién participó y cómo afectó eso a la volatilidad y la duración del ciclo?",

    # BLOQUE 4 — Trampas para detectar alucinaciones
    "¿Cuál será el precio de Bitcoin en el próximo halving? Responde solo con información del contexto proporcionado, sin especular.",
    "¿Qué indicadores técnicos específicos señalaron exactamente el techo del ciclo de 2021 en Ethereum el día 10 de noviembre? Si no tienes esa información en el contexto, indícalo explícitamente.",
    "Según el contexto que tienes disponible, ¿existe algún patrón que permita predecir el momento exacto en que termina la fase de distribución de un ciclo? Si no hay evidencia suficiente, di por qué.",
    "¿Puede el comportamiento del precio de Ethereum en ciclos pasados utilizarse como modelo predictivo fiable para el siguiente ciclo? Lista explícitamente los supuestos que tendrías que asumir para que esa predicción sea válida.",
    "¿Qué información adicional necesitarías, más allá del contexto proporcionado, para hacer una predicción fundamentada sobre el próximo techo de ciclo de Bitcoin o Ethereum?",
]

BLOQUES = {
    "BLOQUE 1 — Hechos históricos verificables": range(0, 5),
    "BLOQUE 2 — Patrones de ciclos y correlaciones": range(5, 10),
    "BLOQUE 3 — Razonamiento con incertidumbre": range(10, 15),
    "BLOQUE 4 — Trampas anti-alucinación": range(15, 20),
}

# ── Cliente Gemini ────────────────────────────────────────────────────────────
client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

# ── Bucle principal ───────────────────────────────────────────────────────────
resultados = []

for i, pregunta in enumerate(preguntas):
    bloque_actual = next(
        nombre for nombre, rango in BLOQUES.items() if i in rango
    )

    print(f"\n{'='*70}")
    print(f"[{bloque_actual}]")
    print(f"PREGUNTA {i+1}/{len(preguntas)}: {pregunta}")
    print("="*70)

    prompt = f"""Eres un analista experto en ciclos de mercado de criptomonedas.
Responde ÚNICAMENTE basándote en el contexto proporcionado.
Si la información no está en el contexto, indícalo explícitamente con:
"⚠️ Esta información no está en el contexto proporcionado."
No especules ni inventes datos, fechas o precios.

<CONTEXTO_ESTATICO>
{contexto}
</CONTEXTO_ESTATICO>

<DINAMICO>
Sin datos dinámicos por ahora. Razona solo con los patrones estáticos.
</DINAMICO>

<NOTICIAS>
Sin noticias por ahora.
</NOTICIAS>

<PREGUNTA>
{pregunta}
</PREGUNTA>
"""

    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )
        respuesta = response.text

    except Exception as e:
        respuesta = f"❌ ERROR: {str(e)}"

    print(respuesta)

    resultados.append({
        "bloque": bloque_actual,
        "numero": i + 1,
        "pregunta": pregunta,
        "respuesta": respuesta,
    })

    # Pausa entre llamadas para respetar el rate limit (5 RPM free tier)
    if i < len(preguntas) - 1:
        print(f"\n⏳ Esperando 13s para respetar rate limit...")
        time.sleep(13)

# ── Guardar resultados en TXT para análisis ───────────────────────────────────



[BLOQUE 1 — Hechos históricos verificables]
PREGUNTA 1/20: ¿En qué año ocurrió el halving de Bitcoin de 2020 y cuánto bajó la recompensa de bloque? ¿Qué efecto histórico tuvo en el precio en los 12 meses siguientes?
Basándome en el contexto proporcionado:

1.  El halving de Bitcoin de 2020 ocurrió el 11 de mayo de 2020 (Patron 1).
2.  La recompensa por bloque que reciben los mineros se dividió a la mitad (Patron 1).
3.  En los 12 meses siguientes, el mercado se encontraba en la "Fase 3 - Bull run post-halving", que históricamente dura entre 12 y 18 meses tras el halving. Para el Ciclo 3 (halving 2020), el rally desde el halving hasta el pico alcanzó aproximadamente +702% en 549 días (nov-2021), lo que implica un movimiento alcista significativo durante los primeros 12 meses (Patron 1).

La evidencia para estos patrones es [MODERADA], basada en 3 ciclos observados (Patron 1).

⏳ Esperando 13s para respetar rate limit...

[BLOQUE 1 — Hechos históricos verificables]
PREGUNTA 2/20: ¿Cuál 

NameError: name 'ou' is not defined